In [1]:
from torchvision.models import mobilenet_v2, MobileNet_V2_Weights
import torch.nn as nn
from torchvision import datasets, transforms, models
import os
from torch.utils.data import DataLoader
import torch
from torch.optim import *
from PIL import Image
import torch.nn.functional as F

In [2]:
weights = MobileNet_V2_Weights.DEFAULT
model_mbvnet = mobilenet_v2(weights=weights)

Downloading: "https://download.pytorch.org/models/mobilenet_v2-7ebf99e0.pth" to C:\Users\HP/.cache\torch\hub\checkpoints\mobilenet_v2-7ebf99e0.pth


100%|██████████| 13.6M/13.6M [00:03<00:00, 3.91MB/s]


In [3]:
# Freeze backbone
for param in model_mbvnet.parameters():
    param.requires_grad = False

In [4]:
# Transforms
transform = weights.transforms()
base_dir = 'Splited_Data'
train_dataset = datasets.ImageFolder(os.path.join(base_dir, 'train'), transform=transform)
val_dataset = datasets.ImageFolder(os.path.join(base_dir, 'val'), transform=transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model_mbvnet.classifier[1].parameters(), lr=0.001)
dataset = datasets.ImageFolder('aug_processed_data', transform=transform)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [5]:
# Replace classifier
num_features = model_mbvnet.classifier[1].in_features
model_mbvnet.classifier[1] = nn.Linear(num_features, len(train_dataset.classes))
model_mbvnet = model_mbvnet.to(device)

In [6]:
# === 3. Training loop ===
def train_model(epochs=10):
    best_val_acc = 0.0
    for epoch in range(epochs):
        model_mbvnet.train()
        total_loss, correct, total = 0, 0, 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model_mbvnet(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

        train_acc = 100 * correct / total
        avg_loss = total_loss / len(train_loader)

        model_mbvnet.eval()
        val_correct, val_total, val_loss = 0, 0, 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model_mbvnet(images)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                _, preds = torch.max(outputs, 1)
                val_correct += (preds == labels).sum().item()
                val_total += labels.size(0)

        val_acc = 100 * val_correct / val_total
        avg_val_loss = val_loss / len(val_loader)
        if val_acc > best_val_acc:
            best_val_acc = val_acc

        print(f"Epoch {epoch+1} | Train Loss: {avg_loss:.4f} | Train Acc: {train_acc:.2f}% | Val Loss: {avg_val_loss:.4f} | Val Acc: {val_acc:.2f}%")

    print(f"\n🏆 Best Val Accuracy: {best_val_acc:.2f}%")

In [7]:
# === Run training ===
train_model(epochs=10)

Epoch 1 | Train Loss: 0.6739 | Train Acc: 64.38% | Val Loss: 0.6589 | Val Acc: 65.00%
Epoch 2 | Train Loss: 0.6777 | Train Acc: 58.12% | Val Loss: 0.6586 | Val Acc: 70.00%
Epoch 3 | Train Loss: 0.6673 | Train Acc: 62.50% | Val Loss: 0.6599 | Val Acc: 72.50%
Epoch 4 | Train Loss: 0.6711 | Train Acc: 58.12% | Val Loss: 0.6514 | Val Acc: 75.00%
Epoch 5 | Train Loss: 0.6641 | Train Acc: 63.12% | Val Loss: 0.6470 | Val Acc: 80.00%
Epoch 6 | Train Loss: 0.6789 | Train Acc: 56.25% | Val Loss: 0.6510 | Val Acc: 77.50%
Epoch 7 | Train Loss: 0.6732 | Train Acc: 58.75% | Val Loss: 0.6497 | Val Acc: 75.00%
Epoch 8 | Train Loss: 0.6742 | Train Acc: 60.00% | Val Loss: 0.6468 | Val Acc: 75.00%
Epoch 9 | Train Loss: 0.6672 | Train Acc: 60.62% | Val Loss: 0.6450 | Val Acc: 75.00%
Epoch 10 | Train Loss: 0.6823 | Train Acc: 56.25% | Val Loss: 0.6431 | Val Acc: 75.00%

🏆 Best Val Accuracy: 80.00%


Model Testing

In [8]:
def predict_single_image(image_path, model, class_names):
    model.eval()
    transform = transforms.Compose([
        transforms.Resize((128, 128)),
        transforms.ToTensor()
    ])

    img = Image.open(image_path).convert("RGB")
    img_tensor = transform(img).unsqueeze(0)  # Add batch dimension

    with torch.no_grad():
        output = model(img_tensor)
        probs = F.softmax(output, dim=1)
        _, predicted = torch.max(probs, 1)

    print(f"Predicted Class: {class_names[predicted.item()]}")
    print(f"Class Probabilities: {probs.squeeze().numpy()}")

In [9]:
# Assuming dataset = ImageFolder(...)
class_names = dataset.classes  # ['healthy', 'infected']

# Path to one test image
test_image_path_1 = "processed_data/serie infected leaves/infected_05.png"

predict_single_image(test_image_path_1, model_mbvnet, class_names)

Predicted Class: series_infected_leaves
Class Probabilities: [0.43912542 0.5608745 ]


Model Evaluation

In [11]:
from sklearn.metrics import classification_report

def evaluate_final_model():
    model_mbvnet.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model_mbvnet(images)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    print("\n📊 Final Evaluation on Validation Set:")
    print(classification_report(all_labels, all_preds, target_names=val_dataset.classes, digits=2))

# Run this after training
print("Evaluation of MobilnetV2 Model")
evaluate_final_model()

Evaluation of MobilnetV2 Model

📊 Final Evaluation on Validation Set:
                        precision    recall  f1-score   support

  serie_healthy_leaves       0.86      0.60      0.71        20
series_infected_leaves       0.69      0.90      0.78        20

              accuracy                           0.75        40
             macro avg       0.77      0.75      0.74        40
          weighted avg       0.77      0.75      0.74        40

